# Multimodal pair verification (simulation)

Train a regularized pair verifier on matching pairs, harder negatives selected by cosine similarity, and random negatives. Compare against cosine similarity across every held-out pair. Report AUROC, average precision and image-to-text Recall@5. Generate noisy paired observations with both shared group structure and shared instance-level information.

In [ ]:
from pathlib import Path
import subprocess, sys
root = Path.cwd() / 'crossmodal-pair-audit-heavy-run'
root.mkdir(exist_ok=True)
subprocess.run([sys.executable,'-m','pip','install',*['numpy==2.2.6', 'scipy==1.15.3', 'scikit-learn==1.7.2', 'joblib==1.6.0']],check=True)

In [ ]:
(root / 'train.py').write_text('import json\nimport os\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import (\n    average_precision_score,\n    roc_auc_score,\n)\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import StandardScaler\n\nROOT = Path(__file__).resolve().parent\nDATA = Path(os.environ.get("PORTFOLIO_DATA", ROOT / "data"))\nSEEDS = [13, 37, 61]\n\n\ndef save(obj, name):\n    folder = ROOT / "checkpoints"\n    folder.mkdir(exist_ok=True)\n    joblib.dump(obj, folder / (name + ".joblib"), compress=3)\n\n\ndef scores(y, p):\n    return {\n        "auroc": float(roc_auc_score(y, p)),\n        "average_precision": float(average_precision_score(y, p)),\n    }\n\n\ndef dataset(seed):\n    rng = np.random.default_rng(seed)\n\n    def sample(n):\n        groups = rng.integers(4, size=n)\n        latent = rng.normal(size=(n, 32))\n        a = latent + rng.normal(0, 1.4, (n, 32))\n        b = latent + rng.normal(0, 1.4, (n, 32))\n        a[:, :8] += groups[:, None]\n        b[:, :8] += groups[:, None]\n        return a, b\n\n    return (*sample(750), *sample(160))\n\n\ndef unit(x):\n    return x / np.maximum(np.linalg.norm(x, axis=1, keepdims=True), 1e-9)\n\n\ndef pair_features(a, b):\n    a, b = unit(a), unit(b)\n    return np.c_[np.sum(a * b, 1), np.abs(a - b), a * b]\n\n\ndef run(seed):\n    a, b, u, v = dataset(seed)\n    rng = np.random.default_rng(seed)\n    ids = np.arange(len(a))\n    np.sum(unit(a) * unit(b), axis=1)\n    cos = unit(a) @ unit(b).T\n    np.fill_diagonal(cos, -np.inf)\n    hard = np.argsort(-cos, axis=1)[:, :5]\n    neg = np.column_stack(\n        [\n            hard[:, 0],\n            hard[:, 2],\n            hard[:, 4],\n            (ids + rng.integers(1, len(a), len(a))) % len(a),\n        ]\n    )\n    n = neg.shape[1]\n    x = np.vstack(\n        [pair_features(a, b), pair_features(np.repeat(a, n, axis=0), b[neg.ravel()])]\n    )\n    y = np.r_[np.ones(len(a)), np.zeros(n * len(a))]\n    model = make_pipeline(\n        StandardScaler(),\n        LogisticRegression(C=0.005, class_weight="balanced", max_iter=400),\n    ).fit(x, y)\n    save(model, f"pair-model-{seed}")\n    reference = np.eye(len(u), dtype=bool).ravel()\n    paired = pair_features(np.repeat(u, len(v), axis=0), np.tile(v, (len(u), 1)))\n    out = {}\n    for method, s in [\n        ("cosine", paired[:, 0]),\n        ("learned", model.decision_function(paired)),\n    ]:\n        matrix = s.reshape(len(u), len(v))\n        out[method] = {\n            **scores(reference, s),\n            "recall_at_5": float(\n                np.mean(\n                    np.any(\n                        np.argsort(-matrix, axis=1)[:, :5]\n                        == np.arange(len(u))[:, None],\n                        axis=1,\n                    )\n                )\n            ),\n        }\n    return out\n\n\nif __name__ == "__main__":\n    runs = {s: run(s) for s in SEEDS}\n    first = next(iter(runs.values()))\n    means = {\n        k: {m: float(np.mean([r[k][m] for r in runs.values()])) for m in v}\n        for k, v in first.items()\n    }\n    (ROOT / "results.json").write_text(\n        json.dumps({"seeds": SEEDS, "runs": runs, "means": means}, indent=2)\n    )\n    print(means)\n',encoding='utf8')
subprocess.run([sys.executable,'train.py'],cwd=root,check=True)

In [ ]:
print((root/'results.json').read_text())